# Symmetric vs. Asymmetric Semantic Search

A critical distinction for your setup is symmetric vs. asymmetric semantic search:

For *symmetric semantic search* your query and the entries in your corpus are of about the same length and have the same amount of content. An example would be searching for similar questions: Your query could for example be “How to learn Python online?” and you want to find an entry like “How to learn Python on the web?”. For symmetric tasks, you could potentially flip the query and the entries in your corpus.


For *asymmetric semantic search*, you usually have a short query (like a question or some keywords) and you want to find a longer paragraph answering the query. An example would be a query like “What is Python” and you want to find the paragraph “Python is an interpreted, high-level and general-purpose programming language. Python’s design philosophy …”. For asymmetric tasks, flipping the query and the entries in your corpus usually does not make sense.

It sounds like *asymmetric semantic search* is the better aproach to searching through the *resume* text. 

In [126]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data/temp_data")

roll_calls_file = DATA_DIR / "parliament" / "roll_calls_resume.csv"
candidate_test_file = DATA_DIR / "candidate_test" / "Kandidattestdata.xlsx"

df_roll_calls = pd.read_csv(roll_calls_file)
df_FV11 = pd.read_excel(candidate_test_file, sheet_name="FV11")
df_FV15 = pd.read_excel(candidate_test_file, sheet_name="FV15")
df_FV19 = pd.read_excel(candidate_test_file, sheet_name="FV19")
df_FV22 = pd.read_excel(candidate_test_file, sheet_name="FV22")

print(df_roll_calls.shape)
print(df_roll_calls.columns.tolist())
print(df_FV11.shape)
print(df_FV11.columns.tolist())

(2128, 12)
['afstemningid', 'sagstrinid', 'kommentar', 'afstemningstypeid', 'dato', 'sagstrintypeid', 'sagid', 'sagstypeid', 'sag_nummer', 'sag_titel', 'sag_titelkort', 'sag_resume']
(14540, 12)
['id', 'Firstname', 'Lastname', 'Candidates.Party', 'Storkreds', 'gender', 'birthdate', 'Question', 'Answer', 'Answer (Text)', 'IsImportant', 'Comment']


In [127]:
df_FV22.columns.tolist()

['id',
 'Candidates.Firstname',
 'Candidates.Lastname',
 'Candidates.Party',
 'Candidates.Area',
 'gender',
 'birthdate',
 'Question',
 'Answer',
 'Answer (text)',
 'IsImportant',
 'Comment']

In [128]:
# We don't need duplicate questions, so we can drop them and reset the index
df_questions_FV22 = (
    df_FV22[["Question"]]
    .drop_duplicates()
    .dropna()
    .reset_index(drop=True)
)

print(df_questions_FV22.shape)

df_questions_FV22.head(20)

(51, 1)


,Question
0,Afgiften på såkaldte hybridbiler bør være den samme som på benzin- og dieselbiler
1,Aldersgrænsen for at købe øl og vin skal hæves til 18 år
2,Asylansøgere bør sendes til et land uden for EU - f.eks. Rwanda - mens deres ansøgning behandles
3,Behandling hos tandlægen bør være gratis for borgerne ligesom andre sundhedsydelser
4,Brugen af ukrudtsmidlet Roundup bør forbydes i landbruget
5,Danmark bør indføre CO2-afgift på flyrejser
6,Danmark bør tage imod flere kvoteflygtninge
7,Danmark skal bruge flere penge på at styrke tog- og busdrift frem for at bygge nye motorveje
8,Danmark skal undersøge muligheden for at udvikle A-kraft som energikilde herhjemme
9,Den såkaldte Arne-pension skal afskaffes


In [129]:
df_questions_FV22["election"] = "FV22"

In [130]:
def get_unique_questions(df, election):
    out = (
        df[["Question"]]
        .drop_duplicates()
        .dropna()
        .reset_index(drop=True)
    )
    
    out["election"] = election
    return out


df_questions_FV11 = get_unique_questions(df_FV11, "FV11")
df_questions_FV15 = get_unique_questions(df_FV15, "FV15")
df_questions_FV19 = get_unique_questions(df_FV19, "FV19")
df_questions_FV22 = get_unique_questions(df_FV22, "FV22")

In [131]:
df_questions = pd.concat(
    [
        df_questions_FV11,
        df_questions_FV15,
        df_questions_FV19,
        df_questions_FV22,
    ],
    ignore_index=True,
)

In [132]:
df_cases = (
    df_roll_calls[
        [
            "sagid",
            "sag_nummer",
            "sag_titel",
            "sag_titelkort",
            "sag_resume",
            "dato",
        ]
    ]
    .drop_duplicates(subset="sagid")
    .reset_index(drop=True)
)

In [133]:
df_cases["text"] = (
    df_cases["sag_titel"].fillna("")
    + " "
    + df_cases["sag_resume"].fillna("")
)

In [134]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "intfloat/multilingual-e5-base"
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4492.12it/s]


In [135]:
# Picking a random question from FV22 to test the embedding model

df_questions.sample(10, random_state=42)
question = df_questions.iloc[73]["Question"]

print(question)

Integrationsydelsen skal hæves


### Embed all parliamentary cases using the asymmetric way

In [136]:
# Full text embeddings 
case_texts = [
    f"passage: {text}"
    for text in df_cases["text"]
]

case_embeddings_texts = model.encode_document(
    case_texts,
    normalize_embeddings=True,
    show_progress_bar=True,
)

Batches:   0%|          | 0/67 [00:00<?, ?it/s]

Batches: 100%|██████████| 67/67 [00:33<00:00,  2.00it/s]


In [142]:
# Only title embeddings
case_titles = [
    f"passage: {title}"
    for title in df_cases["sag_titel"]
]

case_embeddings_titles = model.encode_document(
    case_titles,
    normalize_embeddings=True,
    show_progress_bar=True,
)

Batches: 100%|██████████| 67/67 [00:08<00:00,  8.24it/s]


In [144]:
# Then embed the query question

query_embedding = model.encode_query(
    [f"query: {question}"],
    normalize_embeddings=True,
)

In [145]:
similarities_texts = query_embedding @ case_embeddings_texts.T
similarities_titles = query_embedding @ case_embeddings_titles.T

In [147]:
# Retrieve the top 10 most similar cases of full text embeddings 

import numpy as np

top_k = 10

top_indices = np.argsort(
    similarities_texts[0]
)[::-1][:top_k]

results_text = df_cases.iloc[top_indices].copy()

results_text["similarity"] = similarities_texts[0][top_indices]

results_text[
    [
        "similarity",
        "sag_nummer",
        "sag_titel",
        "sag_resume",
    ]
]

,similarity,sag_nummer,sag_titel,sag_resume
665,0.863359,L 120,"Forslag til lov om ændring af lov om aktiv socialpolitik, integrationsloven, lov om en aktiv beskæftigelsesindsats og ligningsloven. (Nedsættelse af integrationsydelse og omlægning af dansktillæg).","Med lovforslaget nedsættes integrationsydelsen med 3 pct. for at skabe incitament til at tage et arbejde. Samtidig omlægges dansktillægget til en tidsbegrænset danskbonus bestående af seks rater af 1.500 kr. pr. måned. Derudover indføres en danskbonus på 6.000 kr. til personer på integrationsprogrammet, der ikke er omfattet af integrationsydelse."
1326,0.860468,L 202,Forslag til lov om ændring af lov om integrationsgrunduddannelse (igu). (Udvidelse af målgruppen m.v. for integrationsgrunduddannelsen).,"Det foreslås med lovforslaget, at opholdsbetingelsen for integrationsuddannelsesforløb udvides med 5 år, således at flygtninge og familiesammenførte til flygtninge, som har været i Danmark i 0-10 år, fremover får mulighed for at tage et integrationsgrunduddannelsesforløb. Derudover foreslås det, at omfanget af uddannelse i form af amu og sprogkurser udvides med 3 uger, hvorved der samlet set vil være 23 ugers uddannelse i et samlet integrationsgrundsuddannelsesforløb. \n\nLovforslaget udmønter dele af trepartsaftalen af 21. december 2020 mellem regeringen (Socialdemokratiet) og arbejdsmarkedets parter om at udvide og forbedre integrationsuddannelsesforløb.\n\nDet foreslås, at loven træder i kraft den 1. maj 2021, og at de foreslåede regler alene gælder for integrationsgrunduddannelsesforløb, der påbegyndes den 1. maj 2021 eller senere.\n"
765,0.859450,L 231,"Forslag til lov om ændring af udlændingeloven. (Reform af reglerne om ægtefællesammenføring med nyt integrationskrav i stedet for tilknytningskravet, skærpet boligkrav m.v.).","Formålet med lovforslaget er at stramme ægtefællesammenføringsreglerne, bl.a. ved at styrke fokus på, om begge ægtefæller har udsigt til en vellykket integration i Danmark.\n\nMed lovforslaget erstattes det gældende tilknytningskrav med et integrationskrav. Integrationskravet medfører, at ægtefællesammenføring som hovedregel kun kan gives, hvis den herboende ægtefælle har sprogkundskaber, der svarer til Prøve i Dansk 3, og hvis ægtefællerne samlet set opfylder tre ud af fem øvrige betingelser vedrørende sprogkundskaber, erhvervserfaring og uddannelse. \n\nLovforslaget viderefører de hidtidige regler om, at ægtefællesammenføring normalt kun kan gives, hvis den ægtefælle, der kommer hertil, har haft mindst ét lovligt ophold i Danmark. Derudover skærpes de overførte betingelser i reglerne om ægtefællesammenføring, så de svarer til reglerne for tidsubegrænset opholdstilladelse.\n\nLovforslaget indeholder desuden en skærpelse af boligkravet. Boligen må fremover ikke ligge i bestemte boligområder, som optages på boligkravslisten for ægtefællesammenføring. Udlændinge- og integrationsministeren fastsætter de nærmere regler om kriterierne for optagelse af boligområder på listen og offentliggør, hvilke boligområder der er omfattet af boligkravslisten for ægtefællesammenføring. \n\nDen økonomiske sikkerhedsstillelse ved ægtefællesammenføring hæves til 100.000 kr. (i 2018-niveau) mod i dag 50.000 kr. (55.375,27 kr. i 2018-niveau), og reglerne om gradvis nedsættelse af sikkerhedsstillelsen justeres.\n\nDerudover udvides sprogkravet til den ægtefælle, der kommer hertil, så man fremover udover at bestå en danskprøve på A1-niveau inden 6 måneder også skal bestå en danskprøve på A2-niveau eller lignende senest 9 måneder efter tilmelding til folkeregisteret eller fra meddelelse af tilladelse til familiesammenføring.\n\nLovforslaget er en del af udmøntningen af aftalen af 7. februar 2018 mellem regeringen (Venstre, Liberal Alliance og Konservative Folkeparti), Dansk Folkeparti og Socialdemokratiet om stramme, afbalancerede og realistiske regler for ægtefællesammenføring, hvor integrationen er i centrum."
1756,0.859168,L 82,Forslag til lov om ændring af lov om i

In [148]:
# Retrieve the top 10 most similar cases of title embeddings

import numpy as np

top_k = 10

top_indices = np.argsort(
    similarities_titles[0]
)[::-1][:top_k]

results_title = df_cases.iloc[top_indices].copy()
    
results_title["similarity"] = similarities_titles[0][top_indices]

results_title[
    [
        "similarity",
        "sag_nummer",
        "sag_titel",
        "sag_resume",
    ]
]

,similarity,sag_nummer,sag_titel,sag_resume
314,0.858934,L 188,Forslag til lov om integrationsgrunduddannelse (igu).,"Lovforslaget udmønter den del af trepartsaftalen om arbejdsmarkedsintegration mellem regeringen og arbejdsmarkedets parter af 17. marts 2016, der angår oprettelse af en ny integrationsgrunduddannelse (igu). \n\nMed den foreslåede integrationsgrunddannelse får flygtninge og familiesammenførte adgang til både praktisk oplæring på en virksomhed (som aflønnes med elevløn) og til offentligt betalt skoleundervisning med faglig opkvalificering og danskuddannelse og bliver samtidig selvforsørgende i et ustøttet ansættelsesforhold. I forhold til integrationslovens eksisterende virksomhedsrettede redskaber (løntilskud og virksomhedspraktik) udgør integrationsgrunduddannelsen en væsentlig mindre bureaukratisk ordning for både virksomhed, udlænding og kommune, idet virksomhed og udlænding selv aftaler ansættelsesforholdet uden kommunens mellemkomst. Virksomheder, der etablerer og gennemfører integrationsgrunddannelser, bliver tilgodeset med en bonus. Integrationsuddannelsen indføres som en 3-årig forsøgsordning.\n \nAftalen med arbejdsmarkedets parter, som lovforslaget udmønter, forudsætter igangsættelse af integrationsgrunddannelsen pr. 1. juli 2016.\n\n"
315,0.854428,L 189,Forslag til lov om ændring af integrationsloven og forskellige andre love. (Bedre rammer for at modtage og integrere flygtninge og styrket virksomhedsrettet integrationsprogram m.v.).,"Lovforslaget har til formål at udmønte hovedparten af de dele af aftalen »Bedre rammer for at modtage og integrere flygtninge«, som regeringen har indgået med KL, og de dele af »Trepartsaftale om arbejdsmarkedsintegration«, som regeringen har indgået med arbejdsmarkedets parter, som kræver lovændringer. \n\nLovforslaget indeholder bl.a. ændringer af integrationslovens regler om boligplacering af flygtninge og en målretning af programmet mod beskæftigelse i kraft af en mere virksomhedsrettet indsats og ændringer af finansieringsreglerne, så finansieringen målrettes mod de kommuner, som modtager flest flygtninge, og de kommuner og virksomheder, som leverer de bedste integrationsresultater, samt regelforenklinger på en række områder. Desuden er der forslag om ændringer af danskuddannelsesloven, almenboligloven og almenlejeloven. Herudover indeholder lovforslaget ændringer af friskoleloven og lov om efterskoler og frie fagskoler.\n\nDerudover indeholder lovforslaget ændringer af dagtilbudsloven, idet kommunerne med forslaget i en midlertidig periode på 2 år gives øget mulighed for med udgangspunkt i det enkelte barns behov for sprogstimulering selv at fastsætte omfanget af sprogstimuleringen til en nærmere defineret gruppe af tosprogede børn, der ikke går i dagtilbud."
612,0.853388,L 97,Forslag til lov om ændring af integrationsloven og lov om danskuddannelse til voksne udlændinge m.fl. (Forenkling og justering af reglerne om fordeling og boligplacering af flygtninge og præcisering af personkreds for danskuddannelse m.v.).,"Regeringen ønsker med lovforslaget at forenkle kvotefordelingsreglerne til fordeling af flygtninge mellem kommunerne. Ændringerne er foranlediget af de meget store udsving i landstallet, som man har oplevet de senere år. Det har skabt ønske om en enklere fordelingsmodel. De enkelte dele af forslaget har været drøftet med KL.\n\nDerudover ændrer lovforslaget reglerne om boliganvisning efter integrationsloven, så der ikke kan opnås fordele ved at ændre opholdsgrundlag fra familiesammenført til flygtning. Samtidig tydeliggøres det, at en tilflytningskommune ikke er forpligtet til at anvise en bolig, hvis en flygtning fraflytter den kommune, vedkommende er visiteret til. \n\nLovforslaget ændrer samtidig reglerne i lov om danskuddannelse til voksne udlændinge m.fl., så disse sidestilles med S-kursister (nyankomne udenlandske arbejdstagere m.m.). Danskuddannelsen vil således være omfattet af reglerne om betaling af depositum og reglerne om klippekort.\n"
240,0.

# Building a dataset with the top 10 for each candidate questions

In [150]:
question_texts = [
    f"query: {question}"
    for question in df_questions["Question"]
]

question_embeddings = model.encode_query(
    question_texts,
    normalize_embeddings=True,
    show_progress_bar=True,
)

Batches: 100%|██████████| 5/5 [00:01<00:00,  3.37it/s]


In [151]:
similarities = question_embeddings @ case_embeddings_texts.T

similarities.shape

(136, 2128)

In [152]:
# Build the actual evaluation dataset:

import numpy as np
import pandas as pd

top_k = 10

rows = []

for question_idx in range(len(df_questions)):

    scores = similarities[question_idx]

    top_indices = np.argsort(scores)[::-1][:top_k]

    question = df_questions.iloc[question_idx]

    for rank, case_idx in enumerate(top_indices, start=1):

        case = df_cases.iloc[case_idx]

        rows.append({
            "question": question["Question"],
            "election": question["election"],
            "rank": rank,
            "similarity": scores[case_idx],

            "sagid": case["sagid"],
            "sag_nummer": case["sag_nummer"],
            "sag_titel": case["sag_titel"],
            "sag_resume": case["sag_resume"],

            "relevance": None,
            "notes": None,
        })

df_evaluation = pd.DataFrame(rows)

In [154]:
evaluation_file = DATA_DIR / "evaluation" / "e5_top10_relevance.csv"

evaluation_file.parent.mkdir(
    parents=True,
    exist_ok=True
)

df_evaluation.to_csv(
    evaluation_file,
    index=False,
    encoding="utf-8-sig"
)

print(f"Saved to: {evaluation_file}")

Saved to: ..\data\temp_data\evaluation\e5_top10_relevance.csv


Given a candidate-test question, does E5 retrieve parliamentary cases that are actually relevant to the political issue expressed by that question?
I will manually evaluate the data using 0 - 2 score. 0 = Not relevant, 1 = Partially relevant, 2 = Clearly relevant. 